### Agentic RAG

In [1]:
import os
import logging
from typing import TypedDict, List, Optional
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.documents import Document
from langgraph.graph import StateGraph, START, END

In [2]:
os.environ['ANONYMIZED_TELEMETRY'] = 'False'
logging.getLogger('httpx').setLevel(logging.WARNING)

In [3]:
load_dotenv()

llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)
embeddings = OpenAIEmbeddings()

Sample Documents

In [4]:
# Food documents 
food_docs = [
    'Jollof rice is a popular Nigerian one-pot rice dish. It is cooked with tomatoes, peppers, onions, and a blend of spices. The rice is simmered in the tomato base until it absorbs the flavour. Nigerian jollof is famous for its smoky taste, often called "party jollof" because it is served at celebrations. It is usually eaten with fried plantain, chicken, or beef.',
    'Suya is a spicy Nigerian grilled meat skewer. The meat is coated in yaji, a spice mix made from ground peanuts, ginger, and chilli. Suya is traditionally grilled over an open flame and served with onions and tomatoes. It originated from northern Nigeria and is now sold across the country at night markets. Beef, chicken, and ram are the most common types of meat used.',
    'Pounded yam is a smooth, stretchy Nigerian staple made from boiled yam. It is pounded until soft and served with soups like egusi, efo riro, or ogbono. It is eaten by hand, dipped into the soup. Pounded yam is common at traditional ceremonies, especially weddings, where it is served to guests as a sign of hospitality.',
]

# Finance documents 
finance_docs = [
    'GTBank USSD code is *737#. Customers use it to check balance, transfer money, buy airtime, and pay bills. It works on any mobile phone without internet. To transfer money, dial *737*1*Amount*AccountNumber#. To check your balance, dial *737*6*1#. The service is available 24/7 and is free for most basic operations.',
    'Naira transfer limits depend on the customer tier. Tier 1 accounts can transfer up to ₦50,000 per day. Tier 2 up to ₦200,000 per day, and Tier 3 up to ₦5,000,000 per day. Limits apply to mobile and USSD transactions. To increase your tier, you must submit additional KYC documents such as a utility bill and a valid ID to your bank.',
    'To open a bank account in Nigeria, you need a valid ID (NIN, driver\'s licence, or passport), a passport photograph, and proof of address. You can also open a Tier 1 account with just a phone number and NIN. Tier 1 accounts have lower transaction limits but require no paperwork. Most Nigerian banks now allow account opening through their mobile apps in under 10 minutes.',
]

print(f'{len(food_docs)} food docs')
print(f'{len(finance_docs)} finance docs')

3 food docs
3 finance docs


Convert to Document Objects

In [5]:
documents = []

for text in food_docs:
    documents.append(Document(
        page_content=text, 
        metadata={'domain': 'food'}
    ))
    
for text in finance_docs:
    documents.append(Document(
        page_content=text,
        metadata={'domain': 'finance'}
    ))
    
print(f' {len(documents)} Document objects ready')

for doc in documents:
    print(f'[{doc.metadata["domain"]}] {doc.page_content[:100]}...')

 6 Document objects ready
[food] Jollof rice is a popular Nigerian one-pot rice dish. It is cooked with tomatoes, peppers, onions, an...
[food] Suya is a spicy Nigerian grilled meat skewer. The meat is coated in yaji, a spice mix made from grou...
[food] Pounded yam is a smooth, stretchy Nigerian staple made from boiled yam. It is pounded until soft and...
[finance] GTBank USSD code is *737#. Customers use it to check balance, transfer money, buy airtime, and pay b...
[finance] Naira transfer limits depend on the customer tier. Tier 1 accounts can transfer up to ₦50,000 per da...
[finance] To open a bank account in Nigeria, you need a valid ID (NIN, driver's licence, or passport), a passp...


**Build a searchable vector store from the 6 documents and create three retrievers — one for food, one for finance, one for both — so later nodes can fetch relevant docs by meaning and domain.**

Step 3: Build the Vector Store and Three Retrievers

In [6]:
vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embeddings
)

print(f'Vector store built — {vectorstore._collection.count()} chunks')

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Vector store built — 6 chunks


**Create three retriever views over the same store — food-only, finance-only, and unfiltered — so Adaptive RAG can route queries to the right domain.**

Create the Three Retrievers

In [7]:
food_retriever = vectorstore.as_retriever(
    search_kwargs={'k': 2, 'filter': {'domain': 'food'}}
)

finance_retriever = vectorstore.as_retriever(
    search_kwargs={'k': 2, 'filter': {'domain': 'finance'}}
)

both_retriever = vectorstore.as_retriever(
    search_kwargs={'k': 4}
)

print('Three retrievers ready')
print('   food_retriever:    filter=domain:food')
print('   finance_retriever: filter=domain:finance')
print('   both_retriever:    no filter')

Three retrievers ready
   food_retriever:    filter=domain:food
   finance_retriever: filter=domain:finance
   both_retriever:    no filter


**Confirm each retriever returns only the chunks from its own domain, proving the metadata filters work.**

In [8]:
print('--- FOOD retriever ---')
for d in food_retriever.invoke('How do I make jollof rice?'):
    print(f'    {d.page_content[:100]}...')
    
print('\n--- FINANCE retriever ---')
for d in finance_retriever.invoke('What is GTBank USSD?'):
    print(f'    {d.page_content[:100]}...')
    
print('\n--- BOTH retriever ---')
print('\n--- FINANCE retriever ---')
for d in both_retriever.invoke('Nigerian culture'):
    print(f'    [{d.metadata["domain"]}] {d.page_content[:100]}')

--- FOOD retriever ---


Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


    Jollof rice is a popular Nigerian one-pot rice dish. It is cooked with tomatoes, peppers, onions, an...
    Pounded yam is a smooth, stretchy Nigerian staple made from boiled yam. It is pounded until soft and...

--- FINANCE retriever ---
    GTBank USSD code is *737#. Customers use it to check balance, transfer money, buy airtime, and pay b...
    Naira transfer limits depend on the customer tier. Tier 1 accounts can transfer up to ₦50,000 per da...

--- BOTH retriever ---

--- FINANCE retriever ---
    [food] Jollof rice is a popular Nigerian one-pot rice dish. It is cooked with tomatoes, peppers, onions, an
    [food] Pounded yam is a smooth, stretchy Nigerian staple made from boiled yam. It is pounded until soft and
    [food] Suya is a spicy Nigerian grilled meat skewer. The meat is coated in yaji, a spice mix made from grou
    [finance] To open a bank account in Nigeria, you need a valid ID (NIN, driver's licence, or passport), a passp


Create intent_classifier

In [9]:
intent_prompt = ChatPromptTemplate.from_messages([
    ('system',
     'You are a classifier, NOT an answerer.\n'
     '\n'
     'Classify the user question into one of two categories:\n'
     '- RETRIEVE: about Nigerian food (jollof, suya, pounded yam) '
     'or Nigerian finance (GTBank, transfer limits, bank accounts).\n'
     '- DIRECT: greetings, math, general knowledge, chit-chat.\n'
     '\n'
     'Respond with EXACTLY one word: RETRIEVE or DIRECT.\n'
     'Do NOT answer the question. Do NOT explain. Just the word.'),
    ('human', '{query}'),
])

intent_classifier = intent_prompt | llm | StrOutputParser()

**Create the Adaptive RAG domain classifier prompt that routes a query to FOOD, FINANCE, or BOTH.**

Step 5: The Domain Classifier Prompt

In [10]:
domain_prompt =  ChatPromptTemplate.from_messages([
    ('system',
     'Classify the user question into one of three domains:\n'
     '\n'
     '- FOOD: about Nigerian dishes (jollof, suya, pounded yam).\n'
     '- FINANCE: about Nigerian banking (GTBank, transfer limits, accounts).\n'
     '- BOTH: mentions topics from both domains.\n'
     '\n'
     'Answer with exactly one word: FOOD, FINANCE, or BOTH.'),
    ('human', '{query}'),
])

domain_classifier = domain_prompt | llm | StrOutputParser()

Step 5b: Test the Domain Classifier

In [11]:
test_queries = [
    'How do I make jollof rice?',
    'What is GTBank USSD code?',
    'Compare jollof rice and bank accounts',
    'Tell me about suya',
]

for q in test_queries:
    result = domain_classifier.invoke({'query': q}).strip().upper()
    print(f'{result:<10} <- {q}')

FOOD       <- How do I make jollof rice?
FINANCE    <- What is GTBank USSD code?
BOTH       <- Compare jollof rice and bank accounts
FOOD       <- Tell me about suya


**Create the Corrective RAG relevance grader prompt that checks whether each retrieved document is relevant to the query.**

Step 6a: The Relevance Grader

In [12]:
grader_prompt = ChatPromptTemplate.from_messages([
    ('system',
     'Given a query and a document, decide if the document is relevant. '
     'Answer with exactly one word: YES or NO.'),
    ('human', 'Query: {query}\n\nDocument:\n{document}'),
])

relevance_grader = grader_prompt | llm | StrOutputParser()

Step 6b: Test the Grader

In [13]:
food_results = food_retriever.invoke('How do I make jollof rice?')

result1 = relevance_grader.invoke({
    'query': 'How do I make jollof rice?',
    'document': food_results[0].page_content,
}).strip().upper()

# Get real docs from the finance retriever
finance_results = finance_retriever.invoke('What is GTBank USSD?')

result2 = relevance_grader.invoke({
    'query': 'How do I make jollof rice?',
    'document': finance_results[0].page_content,
}).strip().upper()

print(f'Relevant doc   -> {result1}')
print(f'Irrelevant doc -> {result2}')

Relevant doc   -> YES
Irrelevant doc -> NO


In [14]:
food_results

[Document(metadata={'domain': 'food'}, page_content='Jollof rice is a popular Nigerian one-pot rice dish. It is cooked with tomatoes, peppers, onions, and a blend of spices. The rice is simmered in the tomato base until it absorbs the flavour. Nigerian jollof is famous for its smoky taste, often called "party jollof" because it is served at celebrations. It is usually eaten with fried plantain, chicken, or beef.'),
 Document(metadata={'domain': 'food'}, page_content='Pounded yam is a smooth, stretchy Nigerian staple made from boiled yam. It is pounded until soft and served with soups like egusi, efo riro, or ogbono. It is eaten by hand, dipped into the soup. Pounded yam is common at traditional ceremonies, especially weddings, where it is served to guests as a sign of hospitality.')]

**Create the "answer from documents" chain that generates a grounded answer using only retrieved context, so the LLM can't hallucinate.**

Step 8a: The Answer-From-Docs Chain

In [15]:
answer_from_docs_prompt = ChatPromptTemplate.from_messages([
    ('system',
     'Answer the query using ONLY the context below. '
     'If the answer is not in the context, say "I don\'t know".'),
    ('human', 'Context:\n{context}\n\nQuery: {query}'),
])

answer_from_docs = answer_from_docs_prompt | llm | StrOutputParser()


Test Answer-From-Docs

In [16]:
food_results = food_retriever.invoke('What is jollof rice?')

context = '\n\n'.join(d.page_content for d in food_results)

answer = answer_from_docs.invoke({
    'context': context,
    'query': 'What is jollof rice?',
})

print(f'{answer}')

Jollof rice is a popular Nigerian one-pot rice dish cooked with tomatoes, peppers, onions, and a blend of spices. The rice is simmered in the tomato base until it absorbs the flavour. It is famous for its smoky taste and is often served at celebrations, commonly referred to as "party jollof." It is usually eaten with fried plantain, chicken, or beef.


**Define the shared state schema that flows through the entire Agentic RAG graph, so all nodes read and write to the same typed dictionary.**

Step 10: The State Schema

In [17]:
class AgenticRAGState(TypedDict, total=False):
    '''
    State for the full Agentic RAG pipeline.

    Fields:
        query: User's original question
        intent: DIRECT or RETRIEVE
        domain: FOOD, FINANCE, or BOTH
        rewritten_query: Rewritten query (set on retry only)
        retrieved_docs: All chunks from retrieval
        corrected_docs: Chunks that passed grading
        retry_count: How many retries so far
        answer: Final answer to return
    '''
    
    query: str
    intent: str
    domain: str
    rewritten_query: str
    retrieved_docs: List[Document]
    corrected_docs: List[Document]
    retry_count: int
    answer: str
    
print('AgenticRAGState defined')
print(f'   Fields: {list(AgenticRAGState.__annotations__.keys())}')

AgenticRAGState defined
   Fields: ['query', 'intent', 'domain', 'rewritten_query', 'retrieved_docs', 'corrected_docs', 'retry_count', 'answer']


**Create the first node — it reads query from state, classifies it as DIRECT or RETRIEVE using the intent classifier, and writes the result to intent.**

Step 11: The classify_intent_node

In [18]:
def classify_intent_node(state: AgenticRAGState) -> dict:
    '''
    Classify the query as DIRECT or RETRIEVE.

    Reads:  state['query']
    Writes: state['intent']
    '''
    
    intent = intent_classifier.invoke({
        'query': state['query']
    }).strip().upper()
    
    print(f'Intent: {intent}')
    return {'intent': intent}

Step 11b: Test the Node

In [19]:
# Test RETRIEVE
test1 = classify_intent_node({'query': 'What is jollof rice?'})
print(f'   Returned: {test1}\n')

# Test DIRECT
test2 = classify_intent_node({'query': 'What is 10 + 5?'})
print(f'   Returned: {test2}')

Intent: RETRIEVE
   Returned: {'intent': 'RETRIEVE'}

Intent: DIRECT
   Returned: {'intent': 'DIRECT'}


**Create the second node — it reads query from state, classifies it as FOOD, FINANCE, or BOTH using the domain classifier, and writes the result to domain.**

The classify_domain_node

In [20]:
def classify_domain_node(state: AgenticRAGState) -> dict:
    '''Classify the query into FOOD, FINANCE, or BOTH.

    Reads:  state['query']
    Writes: state['domain']
    '''
    
    domain = domain_classifier.invoke({
        'query': state['query']
    }).strip().upper()
    
    print(f'Domain: {domain}')
    return {'domain': domain}

Test the Node

In [21]:
# Test FOOD
test1 = classify_domain_node({'query': 'How do I make jollof rice?'})
print(f'   Returned: {test1}\n')

# Test FINANCE
test2 = classify_domain_node({'query': 'What is GTBank USSD code?'})
print(f'   Returned: {test2}\n')

# Test BOTH
test3 = classify_domain_node({'query': 'Compare jollof rice and bank accounts'})
print(f'   Returned: {test3}')

Domain: FOOD
   Returned: {'domain': 'FOOD'}

Domain: FINANCE
   Returned: {'domain': 'FINANCE'}

Domain: BOTH
   Returned: {'domain': 'BOTH'}


**Create the food retriever node — it reads query from state, fetches chunks from the food-only retriever, and writes them to retrieved_docs.**

Step 13a: The retrieve_food_node

In [22]:
def retrieve_food_node(state: AgenticRAGState) -> dict:
    '''
    Retrieve chunks from the food domain.

    Reads:  state['query'] or state['rewritten_query']
    Writes: state['retrieved_docs']
    '''
    
    query = state.get('rewritten_query') or state['query']
    docs = food_retriever.invoke(query)
    print(f'Retrieved {len(docs)} food docs for: "{query}"')
    return {'retrieved_docs': docs}

In [23]:
test = {'query': 'What is jollof rice?'}
result = retrieve_food_node(test)

print(f'\nReturned {len(result["retrieved_docs"])} docs')
for d in result['retrieved_docs']:
    print(f'   [{d.metadata["domain"]}] {d.page_content[:70]}...')

Retrieved 2 food docs for: "What is jollof rice?"

Returned 2 docs
   [food] Jollof rice is a popular Nigerian one-pot rice dish. It is cooked with...
   [food] Pounded yam is a smooth, stretchy Nigerian staple made from boiled yam...
